In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse

## Goals

Utilise your web scraping skills to gather information about three German cities – Berlin, Hamburg, and Munich – from Wikipedia. You will start by extracting basic information: the country, the latitude and the longitude of each city and then expand to more dynamic data such as the population.

1. Scraping Basic Information

  1.1. Begin by scraping the country, the latitude and the longitude of each city from their respective Wikipedia pages:

 - Berlin: https://en.wikipedia.org/wiki/Berlin
 - Hamburg: https://en.wikipedia.org/wiki/Hamburg
 - Munich: https://en.wikipedia.org/wiki/Munich

  1.2. Once you have scraped the basic information of each city, reflect on the similarities and patterns in accessing them across the three pages. Also, analyse the URLs to identify any commonalities. Make a loop that executes once and retrieves the country, latitude, and longitude for all three cities.

2. Data Organisation

  2.1 Utilise pandas DataFrame to effectively store the extracted information. This DataFrame should have a row for each city, and columns for each type of information (cityname, country, latitude, longitude). If you feel brave, change latitude and longitude into decimal format.

  2.2 Looking ahead (optional): Create a function from the loop and DataFrame to encapsulate the scraping process. This function can be used repeatedly to fetch updated data whenever necessary. It should return a clean, properly formatted DataFrame.


### Try one city

In [47]:
url = "https://en.wikipedia.org/wiki/Berlin"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)
response

<Response [200]>

In [48]:
city_soup = BeautifulSoup(response.content, 'html.parser')

In [26]:
country = city_soup.find("th", string="Country").find_next().get_text()
country

'Germany'

In [34]:
lat = parse(city_soup.find(class_="latitude").get_text())
lon = parse(city_soup.find(class_="longitude").get_text())
lat, lon

(52.519999999999996, 13.405000000000001)

In [49]:
# I overlooked this one! Thanks Mohmammad!
coordinates = city_soup.find("span", class_="geo")
coordinates

<span class="geo" id="mwGbY">52.52000; 13.40500</span>

In [52]:
lat, lon = coordinates.get_text().split("; ")
lat, lon = float(lat), float(lon)
lat, lon

(52.52, 13.405)

### Try the others

In [46]:
url = "https://en.wikipedia.org/wiki/Hamburg"
# url = "https://en.wikipedia.org/wiki/Munich"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    city_soup = BeautifulSoup(response.content, 'html.parser')
    country = city_soup.find("th", string="Country").find_next().get_text()
    print(country)

    lat = parse(city_soup.find(class_="latitude").get_text())
    lon = parse(city_soup.find(class_="longitude").get_text())
    print(lat, lon)

 Germany
53.55 10.0


In [39]:
print(country.strip())

Germany


### Make a function

In [42]:
def city_info(cities):
    info_dict = {
        "name": [],
        "country": [],
        "latitude": [],
        "longitude": []
    }
    
    headers = {'User-Agent': 'Chrome/134.0.0.0'}
    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            country = city_soup.find("th", string="Country").find_next().get_text().strip()
            info_dict["country"].append(country)
        
            lat = parse(city_soup.find(class_="latitude").get_text())
            lon = parse(city_soup.find(class_="longitude").get_text())
            info_dict["latitude"].append(lat)
            info_dict["longitude"].append(lon)

            info_dict["name"].append(city)
        else:
            print(f"Failed to retrieve page for {city}: {response.status_code}")

    return pd.DataFrame(info_dict)

In [43]:
cities = ["Berlin", "Hamburg", "Munich"]

cities_df = city_info(cities)
cities_df

,name,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


## Expansion: Population

  3.1. Expand the scope of your data gathering by extracting the population of a city. This information changes over time, so we might need to add a timestamp.

  3.2. Organise your information in a DataFrame and wrap it in a separate function.

### Try one city

In [53]:
city = cities_df["name"].iloc[0]
url = f"https://en.wikipedia.org/wiki/{city}"
response = requests.get(url, headers=headers)
if response.status_code == 200:
    city_soup = BeautifulSoup(response.content, 'html.parser')

In [57]:
for row in city_soup.find("table").find_all("tr"):
    if row.find(string="Population"):
        population = int(row.find_next("td").get_text().replace(",", ""))
population

1505036

### Try the others

In [58]:
city = cities_df["name"].iloc[1]
# city = cities_df["name"].iloc[2]
url = f"https://en.wikipedia.org/wiki/{city}"
response = requests.get(url, headers=headers)
if response.status_code == 200:
    city_soup = BeautifulSoup(response.content, 'html.parser')

for row in city_soup.find("table").find_all("tr"):
    if row.find(string="Population"):
        population = int(row.find_next("td").get_text().replace(",", ""))
population

1973896

### Make a function

In [59]:
def pop_info(cities_df):
    populations = []
    for city in cities_df["name"]:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Population"):
                    population = int(row.find_next("td").get_text().replace(",", ""))
        populations.append(population)

    return pd.DataFrame({"population": populations, "date_gathered": pd.Timestamp.now().date()})

In [60]:
pops_df = pop_info(cities_df)
pops_df

,population,date_gathered
0,3596999,2026-10-15
1,1973896,2026-10-15
2,1505036,2026-10-15


## Expansion: Additional target cities

  With your robust scraping skills now honed, venture beyond the confines of Germany and explore other cities around the world. While the extraction methodology for German cities may follow a consistent pattern, this may not be the case for cities from different countries. Can you make a function that returns a clean DataFrame of information for cities worldwide?

In [69]:
# comments have been added to lines modified from above
def city_info(cities):
    info_dict = {
        "name": [],
        "country": [],
        "latitude": [],
        "longitude": []
    }
    
    headers = {'User-Agent': 'Chrome/134.0.0.0'}
    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            country_raw = city_soup.find("th", string="Country").find_next('td') # small tweak in .find_next()
            country = country_raw.get_text().strip().split("[")[0] # strip spaces and trailing links
            info_dict["country"].append(country)
        
            lat = parse(city_soup.find(class_="latitude").get_text())
            lon = parse(city_soup.find(class_="longitude").get_text())
            info_dict["latitude"].append(lat)
            info_dict["longitude"].append(lon)

            info_dict["name"].append(city)
        else:
            print(f"Failed to retrieve page for {city}: {response.status_code}")

    return pd.DataFrame(info_dict)

new_cities_df = city_info(["London", "Paris", "Rome"])
new_cities_df

,name,country,latitude,longitude
0,London,England,51.507222,-0.127500
1,Paris,France,48.856667,2.352222
2,Rome,Italy,41.893333,12.482778


In [71]:
# and it still works with the original cities
city_info(["Berlin", "Hamburg", "Munich"])

,name,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


In [70]:
# pop_info seems to work without modification
def pop_info(cities_df):
    populations = []
    for city in cities_df["name"]:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Population"):
                    population = int(row.find_next("td").get_text().replace(",", ""))
        populations.append(population)

    return pd.DataFrame({"population": populations, "date_gathered": pd.Timestamp.now().date()})

new_pops_df = pop_info(new_cities_df)
new_pops_df

,population,date_gathered
0,9122909,2026-10-15
1,2047602,2026-10-15
2,2745062,2026-10-15


These functions may work for the cities we've tried here, but will surely break for others. Webscraping can be a rather iterative process, full of exceptions and workarounds.